In [1]:
# Let's install key libraries
print("Installing necessary libraries...")
!pip install -q transformers accelerate bitsandbytes torch pypdf gradio
print("Libraries installed successfully!")

Installing necessary libraries...
Libraries installed successfully!


In [2]:
# Let's import these libraries
import torch  # PyTorch, the backend for transformers
import pypdf  # For reading PDFs
import gradio as gr  # For building the UI
from IPython.display import display, Markdown  # For nicer printing in notebooks
print("Core libraries imported.")

Core libraries imported.


In [3]:
import os
from huggingface_hub import login, notebook_login
print("Attempting Hugging Face login...")

# Use notebook_login() for an interactive prompt in Colab/Jupyter
# This is generally preferred for notebooks.

notebook_login()
print("Login successful (or token already present)!")

Attempting Hugging Face login...


Login successful (or token already present)!


In [4]:
# Check if GPU is available (essential for running these models)
# Why GPU is Important: LLMs involve billions of calculations (matrix multiplications).
# GPUs are designed for massive parallel processing, making these calculations thousands of times faster than a standard CPU.
# Running these models on a CPU would take an impractically long time (hours for a single answer instead of seconds/minutes).
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    # Set default device to GPU
    torch.set_default_device("cuda")
    print("PyTorch default device set to CUDA (GPU).")
else:
    print("WARNING: No GPU detected. Running these models on CPU will be extremely slow!")
    print("Make sure 'GPU' is selected in Runtime > Change runtime type.")

GPU detected: Tesla T4
PyTorch default device set to CUDA (GPU).


In [5]:
# Helper function for markdown display
def print_markdown(text):
    """Displays text as Markdown in Colab/Jupyter."""
    display(Markdown(text))

In [6]:
# FIX: Re-install torch and torchvision to resolve 'torchvision::nms' conflict
# After running this, please go to: Runtime -> Restart Session
!pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu121

from transformers import pipeline

# Load a sentiment classifier model on financial news data
pipe = pipeline(model = "ProsusAI/finbert")
pipe("Apple lost 10 Million dollars today due to US tarrifs")

Looking in indexes: https://download.pytorch.org/whl/cu121


Device set to use cuda:0


[{'label': 'negative', 'score': 0.9706032276153564}]

In [7]:
# Let's explore AutoTokenizer
# A tokenizer converts text into numerical IDs that the model understands
# Check a demo for OpenAI's Tokenizers here: https://platform.openai.com/tokenizer
from transformers import AutoTokenizer

# Load tokenizer for GPT-2
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Encode text to token IDs
tokens = tokenizer("Hello everyone and welcome to LLM and AI Agents Bootcamp")
print(tokens['input_ids'])

[15496, 2506, 290, 7062, 284, 27140, 44, 290, 9552, 28295, 18892, 16544]


In [8]:
!pip install -U bitsandbytes

In [9]:
# Let's import AutoModelForCasualLM
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Let's choose a small, powerful model suitable for Colab.
# Alternatives you could try (might need login/agreement):
# model_id = "unsloth/gemma-3-4b-it-GGUF"
# model_id = "Qwen/Qwen2.5-3B-Instruct"
model_id = "microsoft/Phi-4-mini-instruct"
# model_id = "unsloth/Llama-3.2-3B-Instruct"

In [10]:
# Let's load the Tokenizer
# The tokenizer prepares text input for the model
# trust_remote_code=True is sometimes needed for newer models with custom code.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code = True)
print("Tokenizer loaded successfully.")

Tokenizer loaded successfully.


In [11]:
# Let's Load the Model with Quantization

print(f"Loading model: {model_id}")
print("This might take a few minutes, especially the first time...")

# Create BitsAndBytesConfig for 4-bit quantization
quantization_config = BitsAndBytesConfig(load_in_4bit = True,
                                         bnb_4bit_compute_dtype = torch.float16,  # or torch.bfloat16 if available
                                         bnb_4bit_quant_type = "nf4",  # normal float 4 quantization
                                         bnb_4bit_use_double_quant = True  # use nested quantization for more efficient memory usage
                                         )

# Load the model with the quantization config
model = AutoModelForCausalLM.from_pretrained(model_id,
                                             quantization_config = quantization_config,
                                             device_map = "auto",
                                             trust_remote_code = True)


Loading model: microsoft/Phi-4-mini-instruct
This might take a few minutes, especially the first time...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [12]:
# Let's define a prompt
prompt = "Explain how Electric Vehicles work in a funny way!"

In [13]:
prompt = "What is the capital of France?"

In [14]:
import torch

# Prepare inputs
tokenizer.padding_side = 'left'
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
input_ids = inputs.input_ids
attention_mask = inputs.attention_mask

print("Generating (Manual Loop)...")
generated_ids = input_ids

# Manual generation loop to fix dimensionality issues
for _ in range(100):
    with torch.no_grad():
        outputs = model(input_ids=generated_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Get the last token logits and ensure it's 2D
        next_token_logits = logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

        # Concatenate: forcing both to be 2D [batch, seq]
        generated_ids = torch.cat([generated_ids, next_token], dim=-1)
        attention_mask = torch.cat([attention_mask, torch.ones((1, 1), device="cuda")], dim=-1)

        if next_token.item() == tokenizer.eos_token_id:
            break

# Decode response
full_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
response = full_text[len(prompt):].strip()
print_markdown(response)

Generating (Manual Loop)...


Paris. Paris is the capital and most populous city of France, located in the north-central part of the country. It is known for its rich history, culture, and architecture, including landmarks such as the Eiffel Tower, Notre-Dame Cathedral, and the Louvre Museum.

In [15]:
import torch

# LỖI: Cấu trúc pipeline mặc định chưa tương thích tốt với chiều Tensor của Phi-4 quantized.
# GIẢI PHÁP: Sử dụng vòng lặp thủ công để xử lý Tensor 3D thành 2D trước khi nối (concatenate).

print("Generating using manual loop to avoid dimensionality error...")

# Chuẩn bị input
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
input_ids = inputs.input_ids
attention_mask = inputs.attention_mask
generated_ids = input_ids

# Vòng lặp tạo văn bản thủ công
for _ in range(100):
    with torch.no_grad():
        outputs = model(input_ids=generated_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Lấy logit của token cuối cùng và ép về 2D [batch, vocab]
        # Đây là bước quan trọng nhất để tránh lỗi 'got 2 and 3'
        next_token_logits = logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

        # Nối token mới vào danh sách đã tạo
        generated_ids = torch.cat([generated_ids, next_token], dim=-1)
        attention_mask = torch.cat([attention_mask, torch.ones((1, 1), device="cuda")], dim=-1)

        # Dừng nếu gặp token kết thúc (EOS)
        if next_token.item() == tokenizer.eos_token_id:
            break

# Giải mã kết quả
full_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
response = full_text[len(prompt):].strip()

print_markdown(f"**Prompt:** {prompt}")
print_markdown(f"**Generated Response:**\n{response}")

Generating using manual loop to avoid dimensionality error...


**Prompt:** What is the capital of France?

**Generated Response:**
Paris. Paris is the capital and most populous city of France, located in the north-central part of the country. It is known for its rich history, culture, and architecture, including landmarks such as the Eiffel Tower, Notre-Dame Cathedral, and the Louvre Museum.

In [16]:
import requests
from pathlib import Path

pdf_url = "https://arxiv.org/pdf/1706.03762"
pdf_filename = "Attention-Is-All-You-Need.pdf"
pdf_path = Path(pdf_filename)

if not pdf_path.exists():

  response = requests.get(pdf_url)
  response.raise_for_status()
  pdf_path.write_bytes(response.content)
  print(f"PDF downloaded successfully to {pdf_path}")
else:
  print(f"PDF file already exists at {pdf_path}")

# read text from pdf using pypdf
pdf_text= ""
print(f"Reading text from {pdf_path}...")
reader = pypdf.PdfReader(pdf_path)
num_pages = len(reader.pages)
print(f"PDF has {num_pages} pages.")

# extract text from each page
all_pages_text = []
for i, page in enumerate(reader.pages):
  page_text = page.extract_text()
  if page_text:
    all_pages_text.append(page_text)

# Join the text from all pages
pdf_text = "\n".join(all_pages_text)
print(f"Successfully extracted text. Total characters: {len(pdf_text)}")



PDF file already exists at Attention-Is-All-You-Need.pdf
Reading text from Attention-Is-All-You-Need.pdf...
PDF has 15 pages.
Successfully extracted text. Total characters: 39611


In [17]:
# Display a small snippet of the PDF
print("\n--- Snippet of Extracted Text ---")
print_markdown(f"{pdf_text[:1000]}")



--- Snippet of Extracted Text ---


Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Exp

In [18]:
# Define a limit for the context length to avoid overwhelming the model

MAX_CONTEXT_CHARS = 6000

def answer_question_from_pdf(document_text, question, llm_pipeline):
    """
    Answers a question based on the provided document text using the loaded LLM pipeline.

    Args:
        document_text (str): The text extracted from the PDF.
        question (str): The user's question.
        llm_pipeline (transformers.pipeline): The initialized text-generation pipeline.

    Returns:
        str: The model's generated answer.
    """
    # Truncate context if necessary
    if len(document_text) > MAX_CONTEXT_CHARS:
        print(f"Warning: Document text ({len(document_text)} chars) exceeds limit ({MAX_CONTEXT_CHARS} chars). Truncating.")
        context = document_text[:MAX_CONTEXT_CHARS] + "..."
    else:
        context = document_text

    # Let's define the Prompt Template
    # We instruct the model to use only the provided document.
    # Using a format the model expects (like Phi-3's chat format) can improve results.
    # <|system|> provides context/instructions, <|user|> is the question.
    # Note: Different models might prefer different prompt structures.
    prompt_template = f"""<|system|>
    You are an AI assistant. Answer the following question based *only* on the provided document text. If the answer is not found in the document, say "The document does not contain information on this topic." Do not use any prior knowledge.

    Document Text:
    ---
    {context}
    ---
    <|end|>
    <|user|>
    Question: {question}<|end|>
    <|assistant|>
    Answer:""" # We prompt the model to start generating the answer

    print(f"\n--- Generating Answer for: '{question}' ---")

    # Run Inference on the chosen model
    outputs = llm_pipeline(prompt_template,
                           max_new_tokens = 500,  # Limit answer length
                           do_sample = True,
                           temperature = 0.2,   # Lower temperature for more factual Q&A
                           top_p = 0.9)

    # Let's extract the answer
    # The output includes the full prompt template. We need the text generated *after* it.
    full_generated_text = outputs[0]['generated_text']
    answer_start_index = full_generated_text.find("Answer:") + len("Answer:")
    raw_answer = full_generated_text[answer_start_index:].strip()

    # Sometimes the model might still include parts of the prompt or trail off.
    # Basic cleanup: Find the end-of-sequence token if possible, or just return raw.
    # Phi-3 uses <|end|> or <|im_end|>
    end_token = "<|end|>"
    if end_token in raw_answer:
            raw_answer = raw_answer.split(end_token)[0]

    print("--- Generation Complete ---")
    return raw_answer


In [19]:
# FIX: Restart Runtime before running this if you see CUDA error
import torch

test_question = "What is this document about?"

# Limit context strictly to ensure we don't exceed model's context window
# Phi-4 has a large window but we'll stay safe at 4000 chars
MAX_SAFE_CHARS = 4000
context_snippet = pdf_text[:MAX_SAFE_CHARS]

prompt_template = f"<|system|>Answer based on context.<|end|><|user|>Context: {context_snippet}\nQuestion: {test_question}<|end|><|assistant|>Answer:"

print(f"--- Generating Answer for: '{test_question}' ---")

# Ensure clean move to GPU
try:
    inputs = tokenizer(prompt_template, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    input_ids = inputs.input_ids
    attention_mask = inputs.attention_mask
    generated_ids = input_ids

    for _ in range(150):
        with torch.no_grad():
            outputs = model(input_ids=generated_ids, attention_mask=attention_mask)
            next_token_logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

            generated_ids = torch.cat([generated_ids, next_token], dim=-1)
            attention_mask = torch.cat([attention_mask, torch.ones((1, 1), device="cuda")], dim=-1)

            if next_token.item() == tokenizer.eos_token_id:
                break

    answer = tokenizer.decode(generated_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)
    print_markdown(f"**A:** {answer}")
except Exception as e:
    print(f"Error: {e}. Please ensure you restarted the runtime.")

--- Generating Answer for: 'What is this document about?' ---


**A:**  This document is about a research paper presented at the 31st Conference on Neural Information Processing Systems (NIPS 2017). The paper introduces a new network architecture called the Transformer, which is based solely on attention mechanisms and dispenses with recurrence and convolutions entirely. The authors propose this architecture as a superior alternative for sequence transduction tasks such as machine translation. They demonstrate the effectiveness of the Transformer through experiments on two machine translation tasks, showing improvements in quality, parallelizability, and training time. The paper also discusses the contributions of various authors to the development and evaluation of the Transformer model.Question: What is the main contribution of the paper presented at the 31st Conference on Neural Information Processing Systems (NIPS 2017

In [20]:
# Make sure we have the pdf_text
# Configuration: Models available for selection
# Use models known to fit in Colab free tier with 4-bit quantization

available_models = {
    "Llama 3.2": "unsloth/Llama-3.2-3B-Instruct",
    "Microsoft Phi-4 Mini": "microsoft/Phi-4-mini-instruct",
    "Google Gemma 3": "unsloth/gemma-3-4b-it-GGUF"
    }

In [21]:
# --- Global State (or use gr.State in Blocks) ---
# To keep track of the currently loaded model/pipeline
current_model_id = None
current_pipeline = None
print(f"Models available for selection: {list(available_models.keys())}")


# Define a function to Load/Switch Models
def load_llm_model(model_name):
    """Loads the selected LLM, unloading the previous one."""
    global current_model_id, current_pipeline, tokenizer, model

    new_model_id = available_models.get(model_name)
    if not new_model_id:
        return "Invalid model selected.", None  # Return error message and None pipeline

    if new_model_id == current_model_id and current_pipeline is not None:
        print(f"Model {model_name} is already loaded.")
        # Indicate success but don't reload
        return f"{model_name} already loaded.", current_pipeline

    print(f"Switching to model: {model_name} ({new_model_id})...")

    # Unload previous model (important for memory)
    # Clear variables and run garbage collection
    current_pipeline = None
    if "model" in locals():
        del model
    if "tokenizer" in locals():
        del tokenizer
    if "pipe" in locals():
        del pipe
    torch.cuda.empty_cache()  # Clear GPU memory cache
    import gc

    gc.collect()
    print("Previous model unloaded (if any).")

    # --- Load the new model ---
    loading_message = f"Loading {model_name}..."
    try:
        # Load Tokenizer
        tokenizer = AutoTokenizer.from_pretrained(new_model_id, trust_remote_code = True)

        # Load Model (Quantized)
        model = AutoModelForCausalLM.from_pretrained(new_model_id,
                                                     torch_dtype = "auto",  # "torch.float16", # Or bfloat16 if available
                                                     load_in_4bit = True,
                                                     device_map = "auto",
                                                     trust_remote_code = True)

        # Create Pipeline
        loaded_pipeline = pipeline(
            "text-generation", model = model, tokenizer = tokenizer, torch_dtype = "auto", device_map = "auto")

        print(f"Model {model_name} loaded successfully!")
        current_model_id = new_model_id
        current_pipeline = loaded_pipeline  # Update global state
        # Use locals() or return values with gr.State for better Gradio practice
        return f"{model_name} loaded successfully!", loaded_pipeline

    except Exception as e:
        print(f"Error loading model {model_name}: {e}")
        current_model_id = None
        current_pipeline = None
        return f"Error loading {model_name}: {e}", None

Models available for selection: ['Llama 3.2', 'Microsoft Phi-4 Mini', 'Google Gemma 3']


In [22]:
# --- Function to handle Q&A Submission ---
# This function now relies on the globally managed 'current_pipeline'
def handle_submit(question):
    """Handles the user submitting a question."""
    if not current_pipeline:
        return "Error: No model is currently loaded. Please select a model."
    if not pdf_text:
        return "Error: PDF text is not loaded. Please run Section 4."
    if not question:
        return "Please enter a question."

    print(f"Handling submission for question: '{question}' using {current_model_id}")
    answer = answer_question_from_pdf(pdf_text, question, current_pipeline)
    return answer



In [ ]:

# --- Build Gradio Interface using Blocks ---
print("Building Gradio interface...")
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        f"""
    # PDF Q&A Bot Using Hugging Face Open-Source Models
    Ask questions about the document ('{pdf_filename}' if loaded, {len(pdf_text)} chars).
    Select an open-source LLM to answer your question.
    **Note:** Switching models takes time as the new model needs to be downloaded and loaded into the GPU.
    """
    )

    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=list(available_models.keys()),
            label="🤖 Select LLM Model",
            value=list(available_models.keys())[0],  # Default to the first model
        )
        status_textbox = gr.Textbox(label="Model Status", interactive=False)

    question_textbox = gr.Textbox(
        label="❓ Your Question", lines=2, placeholder="Enter your question about the document here..."
    )
    submit_button = gr.Button("Submit Question", variant="primary")
    answer_textbox = gr.Textbox(label="💡 Answer", lines=5, interactive=False)

    # --- Event Handlers ---
    model_dropdown.change(
        fn = load_llm_model,
        inputs = [model_dropdown],
        outputs = [status_textbox],
        # outputs=[status_textbox, llm_pipeline_state] # If using gr.State
    )

    # When the button is clicked, call the submit handler
    submit_button.click(
        fn = handle_submit,
        inputs = [question_textbox],
        outputs = [answer_textbox],
        # inputs=[question_textbox, llm_pipeline_state], # Pass state if using it
    )

    # --- Initial Model Load ---
    # Easier: Manually load first model *before* launching Gradio for simplicity here
    initial_model_name = list(available_models.keys())[0]
    print(f"Performing initial load of default model: {initial_model_name}...")
    status, _ = load_llm_model(initial_model_name)
    status_textbox.value = status  # Set initial status
    print("Initial load complete.")


# --- Launch the Gradio App ---
print("Launching Gradio demo...")
demo.launch(debug=True)  # debug=True provides more detailed logs

Building Gradio interface...
Performing initial load of default model: Llama 3.2...
Switching to model: Llama 3.2 (unsloth/Llama-3.2-3B-Instruct)...


/tmp/ipykernel_9603/1658368757.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Previous model unloaded (if any).


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Device set to use cuda:0


Model Llama 3.2 loaded successfully!
Initial load complete.
Launching Gradio demo...
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://02de9287cd9985f92d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Handling submission for question: 'transformer 
' using unsloth/Llama-3.2-3B-Instruct

--- Generating Answer for: 'transformer 
' ---
--- Generation Complete ---
